# Map & Environment Preflight (without TomTom calls)


In [1]:
import sys
from pathlib import Path
import ipyleaflet, ipywidgets
from routing.config import PROJECT_ROOT

print("Python executable:", sys.executable)
print("Python venv prefix:", sys.prefix)
print("Project root:", PROJECT_ROOT)
print("Expected venv:", PROJECT_ROOT / ".venv")
print("Same local venv:", Path(sys.prefix).resolve() == (PROJECT_ROOT / ".venv").resolve())
print("ipyleaflet:", ipyleaflet.__version__)
print("ipywidgets:", ipywidgets.__version__)

Python executable: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Python venv prefix: /Users/mojtabashabani/wiser/mary/.venv
Project root: /Users/mojtabashabani/wiser/AiWithPython/mary
Expected venv: /Users/mojtabashabani/wiser/AiWithPython/mary/.venv
Same local venv: False
ipyleaflet: 0.20.0
ipywidgets: 8.1.9


## A. Frontend widget test


In [2]:
from IPython.display import display
from ipywidgets import IntSlider

display(IntSlider(description="Widgets OK"))


IntSlider(value=0, description='Widgets OK')

In [4]:
from ipyleaflet import Map
from ipywidgets import Layout

small_map = Map(center=(51.4994, -0.1760), zoom=14,
                layout=Layout(height="350px", width="100%"))
display(small_map)

Map(center=[51.4994, -0.176], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

## B. OSM network check


In [5]:
import networkx as nx
import math
from routing.map_loader import load_road_graph

G = load_road_graph()
assert isinstance(G, nx.MultiDiGraph) and G.is_directed()
assert G.number_of_nodes() > 0 and G.number_of_edges() > 0
assert all(math.isfinite(float(d["length"])) and float(d["length"]) >= 0
           for _,_,_,d in G.edges(keys=True, data=True))
print("PASS OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")

PASS OSM graph: 747 nodes, 1718 directed edges


## C. Origin & destination interaction



In [6]:
from routing.config import ORIGIN, DESTINATION
from routing.map_view import interactive_map

widget, state = interactive_map(G, origin=ORIGIN,
                                destination=DESTINATION,
                                max_snap_distance_m=350.0)
initial_origin = state["origin_node"]
initial_destination = state["destination_node"]
display(widget)
print("Initial origin:", initial_origin, "Destination:", initial_destination)

Initial origin: 25291778 Destination: 26389375


In [7]:
print("Origin:", state["origin_node"])
print("Destination:", state["destination_node"])
print("Clicked:", state["last_click_latlon"])
assert state["origin_node"] in G
assert state["destination_node"] in G
assert state["destination_node"] == initial_destination
assert state["last_click_latlon"] is not None, "Double-click has not been received by Python"
print("PASS interactive map")

Origin: 25291778
Destination: 26389375
Clicked: None


AssertionError: Double-click has not been received by Python

## D. Traffic pipeline offline smoke-test




In [8]:
from routing.traffic_pipeline import collect_traffic_snapshot, make_routing_graph

class ForbiddenNetworkClient:
    def fetch_flow(self, *args, **kwargs):
        raise AssertionError("Unexpected network request")

snapshot, summary = collect_traffic_snapshot(
    G, ForbiddenNetworkClient(), max_requests=0, pause_s=0
)
assert summary["requests_made"] == 0
assert summary["counts"]["missing"] == G.number_of_edges()
assert summary["usable_observed_edges"] == 0

time_graph = make_routing_graph(snapshot, objective="time", scenario="measured")
distance_graph = make_routing_graph(snapshot, objective="distance", scenario="measured")
assert time_graph.graph["routing_weight"] == "travel_time_s"
assert distance_graph.graph["routing_weight"] == "length"
print("PASS offline traffic preparation; NO TomTom requests made")

PASS offline traffic preparation; NO TomTom requests made
